# 🧠 NeuroEvasion — Training on Google Colab

> **Train both DQN agents on TPU v5e or T4 GPU, then download the weights to your local machine.**

### Supported Accelerators
| Accelerator | Runtime Setting | Device String |
|-------------|----------------|---------------|
| **TPU v5e** | Runtime → Change type → TPU v2 | `xla` (via `torch_xla`) |
| **T4 GPU** | Runtime → Change type → T4 GPU | `cuda` |
| **CPU** | Default | `cpu` (slow, not recommended) |

---

## Step 1 — Detect Accelerator

In [ ]:
import torch

DEVICE = "cpu"  # Default fallback

# --- Try TPU first ---
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    DEVICE = str(xm.xla_device())
    print(f"✅ TPU Available!")
    print(f"   Device: {DEVICE}")
    print(f"   PyTorch: {torch.__version__}")
    print(f"   torch_xla: {torch_xla.__version__}")
except ImportError:
    print("ℹ️  torch_xla not found, checking for GPU...")

# --- Try GPU ---
if DEVICE == "cpu" and torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU Available: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"   PyTorch: {torch.__version__}")
    print(f"   CUDA: {torch.version.cuda}")

if DEVICE == "cpu":
    print("⚠️  No accelerator detected! Training will be very slow.")
    print("   Go to: Runtime → Change runtime type → TPU or GPU")

print(f"\n🎯 Using device: {DEVICE}")

## Step 2 — Clone Repository

In [ ]:
import os

# =============================================
#   SET YOUR GITHUB REPO URL HERE
# =============================================
REPO_URL = "https://github.com/YOUR_USERNAME/NeuroEvasion.git"
# =============================================

if os.path.exists('/content/NeuroEvasion'):
    print("📂 NeuroEvasion already cloned, pulling latest...")
    !cd /content/NeuroEvasion && git pull
else:
    print(f"📥 Cloning {REPO_URL}...")
    !git clone {REPO_URL} /content/NeuroEvasion

os.chdir('/content/NeuroEvasion')
print(f"\n📂 Working directory: {os.getcwd()}")
!ls -la

## Step 3 — Install Dependencies

In [ ]:
# PyTorch and torch_xla are pre-installed on Colab TPU runtimes
# We only need pygame (for module imports) and tensorboard
!pip install pygame tensorboard -q
print("✅ Dependencies installed")

## Step 4 — Patch Device for TPU Compatibility

Our codebase uses `"cuda"` or `"cpu"` as device strings. For TPU, we need to pass the XLA device.
This cell monkey-patches the config so the trainer uses the correct device automatically.

In [ ]:
import sys
sys.path.insert(0, '/content/NeuroEvasion')

from config import Config
from environment.env import NeuroEvasionEnv
from agents.dqn_agent import DQNAgent
from utils import set_global_seed

# Quick sanity check — create an agent on the detected device
config = Config()
env = NeuroEvasionEnv(config)

# For TPU: we need to handle the device differently
if 'xla' in DEVICE.lower():
    import torch_xla.core.xla_model as xm
    test_device = xm.xla_device()
else:
    test_device = DEVICE

agent = DQNAgent(
    in_channels=env.obs_channels,
    grid_size=config.game.grid_size,
    num_actions=env.snake_num_actions,
    config=config.agent,
    device=str(test_device)
)

# Test forward pass
snake_obs, _ = env.reset()
action = agent.select_action(snake_obs)
print(f"✅ Agent created on: {next(agent.policy_net.parameters()).device}")
print(f"✅ Forward pass works! Action: {action}")
print(f"   Params: {sum(p.numel() for p in agent.policy_net.parameters()):,}")

## Step 5 — 🚀 Train!

| Episodes | GPU (T4) | TPU (v5e) | Quality |
|----------|----------|-----------|----------|
| 10,000 | ~5-10 min | ~3-5 min | Basic behaviors |
| 50,000 | ~30-60 min | ~15-30 min | Decent strategies |
| 200,000 | ~2-4 hrs | ~1-2 hrs | Strong co-evolution |
| 500,000 | ~6-10 hrs | ~3-5 hrs | Full convergence |

In [ ]:
# ==========================================
#   CONFIGURE YOUR TRAINING RUN
# ==========================================

EPISODES = 50_000       # Adjust based on your time
GRID_SIZE = 20          # 20×20 grid
LEARNING_RATE = 1e-4
BATCH_SIZE = 128        # TPU benefits from larger batches
SEED = 42

# ==========================================

In [ ]:
import time
from training.logger import TrainingLogger

# --- Setup ---
set_global_seed(SEED)

config = Config()
config.training.num_episodes = EPISODES
config.game.grid_size = GRID_SIZE
config.agent.learning_rate = LEARNING_RATE
config.agent.batch_size = BATCH_SIZE
config.seed = SEED

env = NeuroEvasionEnv(config)

# Resolve device
if 'xla' in DEVICE.lower():
    import torch_xla.core.xla_model as xm
    device_str = str(xm.xla_device())
else:
    device_str = DEVICE

snake_agent = DQNAgent(
    in_channels=env.obs_channels,
    grid_size=GRID_SIZE,
    num_actions=env.snake_num_actions,
    config=config.agent,
    device=device_str,
)
bait_agent = DQNAgent(
    in_channels=env.obs_channels,
    grid_size=GRID_SIZE,
    num_actions=env.bait_num_actions,
    config=config.agent,
    device=device_str,
)

logger = TrainingLogger(f"logs/run_{int(time.time())}")

print(f"🧠 NeuroEvasion — Co-Evolutionary Training")
print(f"{'=' * 60}")
print(f"  Device:       {device_str}")
print(f"  Episodes:     {EPISODES:,}")
print(f"  Grid:         {GRID_SIZE}×{GRID_SIZE}")
print(f"  Batch size:   {BATCH_SIZE}")
print(f"  LR:           {LEARNING_RATE}")
print(f"{'=' * 60}\n")

# --- Training Loop ---
snake_wins = 0
total_games = 0
start_time = time.time()

for episode in range(1, EPISODES + 1):
    snake_obs, bait_obs = env.reset()
    ep_snake_r = 0.0
    ep_bait_r = 0.0
    ep_s_loss = 0.0
    ep_b_loss = 0.0
    loss_count = 0
    step = 0

    for step in range(config.game.max_steps):
        snake_action = snake_agent.select_action(snake_obs)
        bait_action = bait_agent.select_action(bait_obs)

        (s_obs_next, b_obs_next, s_r, b_r, done, info) = env.step(snake_action, bait_action)

        snake_agent.store_transition(snake_obs, snake_action, s_r, s_obs_next, done)
        bait_agent.store_transition(bait_obs, bait_action, b_r, b_obs_next, done)

        s_loss = snake_agent.train_step()
        b_loss = bait_agent.train_step()

        # TPU: mark_step tells XLA to execute queued operations
        if 'xla' in device_str.lower():
            xm.mark_step()

        if s_loss is not None:
            ep_s_loss += s_loss
            ep_b_loss += (b_loss or 0)
            loss_count += 1

        ep_snake_r += s_r
        ep_bait_r += b_r
        snake_obs, bait_obs = s_obs_next, b_obs_next

        if done:
            break

    total_games += 1
    if info.get('event') == 'capture':
        snake_wins += 1

    avg_s_loss = ep_s_loss / max(loss_count, 1)
    avg_b_loss = ep_b_loss / max(loss_count, 1)

    if episode % config.agent.target_sync_interval == 0:
        snake_agent.sync_target_network()
        bait_agent.sync_target_network()

    logger.log_episode(episode, ep_snake_r, ep_bait_r, avg_s_loss, avg_b_loss,
                       info.get('event', 'unknown'), step + 1, snake_agent.epsilon)

    if episode % config.training.log_interval == 0:
        win_rate = snake_wins / max(total_games, 1) * 100
        elapsed = time.time() - start_time
        eps_per_sec = episode / elapsed
        eta_min = (EPISODES - episode) / eps_per_sec / 60
        print(
            f"Ep {episode:>7,d} | "
            f"ε={snake_agent.epsilon:.3f} | "
            f"🐍={ep_snake_r:>7.2f} | "
            f"🎯={ep_bait_r:>7.2f} | "
            f"Win%={win_rate:>5.1f}% | "
            f"{eps_per_sec:.1f} ep/s | "
            f"ETA {eta_min:.0f}m"
        )
        snake_wins = 0
        total_games = 0

    if episode % config.training.checkpoint_interval == 0:
        snake_agent.save(f"checkpoints/snake_ep{episode}.pt")
        bait_agent.save(f"checkpoints/bait_ep{episode}.pt")
        print(f"  💾 Checkpoint saved at ep {episode:,d}")

# Final save
snake_agent.save("checkpoints/snake_final.pt")
bait_agent.save("checkpoints/bait_final.pt")
logger.close()

total_time = time.time() - start_time
print(f"\n✅ Training complete in {total_time/60:.1f} minutes!")
print(f"   Final models: checkpoints/snake_final.pt, checkpoints/bait_final.pt")

## Step 6 — Evaluate

In [ ]:
from evaluation.evaluator import evaluate

results = evaluate(Config(), 'checkpoints/snake_final.pt',
                   'checkpoints/bait_final.pt', num_games=500)

## Step 7 — TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

## Step 8 — 📥 Download Weights

Download to your local machine, then run:
```bash
python main.py demo --snake-model checkpoints/snake_final.pt --bait-model checkpoints/bait_final.pt
```

In [ ]:
from google.colab import files

files.download('checkpoints/snake_final.pt')
files.download('checkpoints/bait_final.pt')
print("✅ Weights downloaded!")

In [ ]:
# OPTIONAL: Download ALL checkpoints as a zip
import shutil
from google.colab import files

shutil.make_archive('neuroevasion_weights', 'zip', '.', 'checkpoints')
files.download('neuroevasion_weights.zip')
print("✅ All checkpoints downloaded")

---
## 🏠 On Your Local Machine

```bash
cd NeuroEvasion && source .venv/bin/activate

# Watch trained agents play
python main.py demo \
  --snake-model checkpoints/snake_final.pt \
  --bait-model checkpoints/bait_final.pt \
  --speed 8

# Evaluate
python main.py evaluate \
  --snake-model checkpoints/snake_final.pt \
  --bait-model checkpoints/bait_final.pt
```
> Demo/evaluate only run forward passes — no GPU needed locally.